## Implemetation of the single head slef transfomer

In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelfAttention(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        # We do NOT use embeddings directly.
        # Transformer first projects tokens into 3 different "views"
        # Each view represents a different role in communication.

        # Query = what this word is asking for
        self.W_q = nn.Linear(d_model, d_model)

        # Key = what this word offers
        self.W_k = nn.Linear(d_model, d_model)

        # Value = actual information/content of the word
        self.W_v = nn.Linear(d_model, d_model)


    def forward(self, x):
        # x shape: (batch_size, sequence_length, d_model)
        # Example: (1, 5, 512) = 1 sentence, 5 words, 512-dim embedding

        # ---- Step 1: Create Q, K, V ----
        # Each word now has three representations:
        # asking vector, matching vector, information vector
        Q = self.W_q(x)   # (batch, seq, d_model)
        K = self.W_k(x)   # (batch, seq, d_model)
        V = self.W_v(x)   # (batch, seq, d_model)


        # ---- Step 2: Similarity score (who should talk to whom?) ----
        # Dot product compares every word with every other word
        # Result: attention score matrix
        # shape -> (batch, seq, seq)
        scores = torch.matmul(Q, K.transpose(-2, -1))


        # ---- Step 3: Scaling ----
        # Without scaling:
        # large vector dimension → huge dot products → softmax saturation
        # saturation → gradients vanish → model stops learning
        d_k = K.size(-1)
        scores = scores / (d_k ** 0.5)


        # ---- Step 4: Softmax normalization ----
        # Convert raw similarity into probability distribution
        # Each row now sums to 1
        # Meaning: how much attention a word gives to every other word
        attention_weights = F.softmax(scores, dim=-1)


        # ---- Step 5: Weighted information aggregation ----
        # Each word collects information from other words
        # based on attention probabilities
        output = torch.matmul(attention_weights, V)

        # output shape: (batch, seq, d_model)
        # Now each word vector contains CONTEXT (not just its own meaning)

        return output, attention_weights

In [3]:

# pretend this is a sentence embedding
# batch=1 sentence
# seq_len=4 words
# d_model=8 features per word
x = torch.randn(1, 4, 8)

print("Input shape:", x.shape)


model = SelfAttention(d_model=8)

output, weights = model(x)

print("\nOutput shape:", output.shape)
print("\nAttention matrix shape:", weights.shape)
print("\nAttention matrix:\n", weights)

Input shape: torch.Size([1, 4, 8])

Output shape: torch.Size([1, 4, 8])

Attention matrix shape: torch.Size([1, 4, 4])

Attention matrix:
 tensor([[[0.2429, 0.2792, 0.2355, 0.2424],
         [0.2539, 0.2531, 0.2898, 0.2032],
         [0.2168, 0.2832, 0.2387, 0.2613],
         [0.2515, 0.1939, 0.3529, 0.2017]]], grad_fn=<SoftmaxBackward0>)
